In [1]:
import pandas as pd

df = pd.read_csv("../MultiagentSystem/predictions_results.csv")
df

,forecast_start_date,y_predict,y_predict_confidence,start_date_price,btc_bybit_close_price,btc_bybit_high_price,btc_bybit_low_price,y_true
0,2026-04-16,LONG,3,74812.8,75149.4,75537.9,73292.0,LONG
1,2026-04-15,LONG,4,74144.2,74812.9,75433.3,73518.0,LONG
2,2026-04-14,LONG,3,74415.1,74144.4,76050.0,73808.3,LONG
3,2026-04-13,LONG,4,70742.7,74415.6,74895.7,70571.4,SHORT
4,2026-04-12,SHORT,-1,73048.9,70742.7,73140.4,70504.4,LONG
...,...,...,...,...,...,...,...,...
95,2026-01-11,LONG,3,90514.8,91004.0,91288.9,90234.0,LONG
96,2026-01-10,SHORT,-2,90643.0,90514.7,90834.5,90395.0,LONG
97,2026-01-09,SHORT,-2,91104.7,90643.1,92024.5,89696.0,SHORT
98,2026-01-08,SHORT,-2,91370.3,91104.7,91676.6,89307.5,SHORT


In [2]:
import pandas as pd

df = pd.read_csv(
    "../MultiagentSystem/predictions_results.csv",
    parse_dates=["forecast_start_date"],
)
df = df.sort_values("forecast_start_date").reset_index(drop=True)

df["price_change"] = df["start_date_price"] - df["btc_bybit_close_price"]
df["volatility_in_low_in_percent"] = abs(df["start_date_price"] - df["btc_bybit_low_price"]) / df["start_date_price"] * 100
df["volatility_in_high_in_percent"] = abs(df["start_date_price"] - df["btc_bybit_high_price"]) / df["start_date_price"] * 100

TEST  = df.tail(30).copy()
TRAIN = df.iloc[:70].copy()

print("TRAIN:", TRAIN["forecast_start_date"].min().date(), "->",
                TRAIN["forecast_start_date"].max().date(), len(TRAIN))
print("TEST :", TEST["forecast_start_date"].min().date(), "->",
                TEST["forecast_start_date"].max().date(), len(TEST))

TRAIN: 2026-01-07 -> 2026-03-17 70
TEST : 2026-03-18 -> 2026-04-16 30


In [3]:
TEST[["y_predict", "y_true", "price_change", "start_date_price", "volatility_in_low_in_percent", "volatility_in_high_in_percent"]].head(30)

,y_predict,y_true,price_change,start_date_price,volatility_in_low_in_percent,volatility_in_high_in_percent
70,NaN,SHORT,2658.0,73906.4,4.615432,1.034011
71,NaN,LONG,1325.1,71248.4,3.451867,0.524924
72,SHORT,SHORT,-586.7,69923.2,0.764124,2.070414
73,SHORT,SHORT,1590.2,70509.9,2.772235,0.836336
74,SHORT,LONG,1057.7,68919.7,2.277578,0.965471
75,LONG,SHORT,-3036.7,67862.0,0.615072,5.820341
76,LONG,LONG,337.2,70898.7,2.820221,0.721734
77,LONG,SHORT,-777.2,70559.5,0.203374,2.088408
78,SHORT,SHORT,2516.6,71336.7,4.466985,0.150834
79,SHORT,SHORT,2412.0,68820.2,4.753401,0.515982


In [10]:
general_budget = 1000.0
once_bet = 50.0

for index, row in TRAIN.iterrows():
    if row["y_predict"] is None or row["y_predict"] == "":
        continue

    start_price = row["start_date_price"]
    close_price = row["btc_bybit_close_price"]
    if start_price is None or close_price is None:
        continue

    # Комиссия на вход
    entry_commission = once_bet * 0.001
    general_budget -= once_bet + entry_commission

    # Считаем P&L в зависимости от направления ставки
    if row["y_predict"] == "LONG":
        # Купили по open, продали по close
        pnl = once_bet * (close_price / start_price - 1)
    else:  # SHORT
        # Шортим: зарабатываем на падении
        pnl = once_bet * (1 - close_price / start_price)

    # Возвращаем ставку + прибыль (или - убыток), минус комиссия на выход
    exit_value = once_bet + pnl
    exit_commission = abs(exit_value) * 0.001
    general_budget += exit_value - exit_commission

print(f"Итого: ${general_budget:.2f}")


Итого: $1012.16


In [39]:
general_budget = 1000.0
once_bet = 50.0
stop_loss_pct = 1.5

for index, row in TEST.iterrows():
    if row["y_predict"] is None or row["y_predict"] == "":
        continue

    start_price = row["start_date_price"]
    close_price = row["btc_bybit_close_price"]
    if start_price is None or close_price is None:
        continue

    # Комиссия на вход
    entry_commission = once_bet * 0.001
    general_budget -= once_bet + entry_commission

    if row["y_predict"] == "LONG":
        # Проверяем: цена падала внутри дня ниже стоп-лосса?
        if row["volatility_in_low_in_percent"] >= stop_loss_pct:
            # Выбило по стопу — фиксируем убыток
            pnl = once_bet * (-stop_loss_pct / 100)
        else:
            pnl = once_bet * (close_price / start_price - 1)
    else:  # SHORT
        # Проверяем: цена росла внутри дня выше стоп-лосса?
        if row["volatility_in_high_in_percent"] >= stop_loss_pct:
            # Выбило по стопу
            pnl = once_bet * (-stop_loss_pct / 100)
        else:
            pnl = once_bet * (1 - close_price / start_price)

    # Возвращаем ставку + прибыль (или - убыток), минус комиссия на выход
    exit_value = once_bet + pnl
    exit_commission = abs(exit_value) * 0.001
    general_budget += exit_value - exit_commission

print(f"Итого: ${general_budget:.2f}")


Итого: $998.14


In [9]:
general_budget = 1000.0
once_bet = 50.0
stop_loss_pct = 2.5
take_profit_pct = 4.0

for index, row in TEST.iterrows():
    if row["y_predict"] is None or row["y_predict"] == "":
        continue

    start_price = row["start_date_price"]
    close_price = row["btc_bybit_close_price"]
    if start_price is None or close_price is None:
        continue

    entry_commission = once_bet * 0.001
    general_budget -= once_bet + entry_commission

    if row["y_predict"] == "LONG":
        if row["volatility_in_low_in_percent"] >= stop_loss_pct:
            pnl = once_bet * (-stop_loss_pct / 100)
        elif row["volatility_in_high_in_percent"] >= take_profit_pct:
            pnl = once_bet * (take_profit_pct / 100)
        else:
            pnl = once_bet * (close_price / start_price - 1)
    else:  # SHORT
        if row["volatility_in_high_in_percent"] >= stop_loss_pct:
            pnl = once_bet * (-stop_loss_pct / 100)
        elif row["volatility_in_low_in_percent"] >= take_profit_pct:
            pnl = once_bet * (take_profit_pct / 100)
        else:
            pnl = once_bet * (1 - close_price / start_price)

    exit_value = once_bet + pnl
    exit_commission = abs(exit_value) * 0.001
    general_budget += exit_value - exit_commission

print(f"Итого: ${general_budget:.2f}")


Итого: $1015.85
